# S0 — Resolve data sources

Stage 0 of the Manhattan Sidewalk Shade Index pipeline.

Resolves every dataset in `CLAUDE.md` §3 against **live** catalogs — no hardcoded/guessed dataset IDs — verifies column names against the published data dictionary, and writes the resolved endpoints to `data/SOURCES.md`.

Two resolution paths, per CLAUDE.md §3:
- **Socrata** (NYC Open Data): street trees (primary + alt), borough boundary — resolved via the Socrata Catalog API by title, then schema-checked.
- **DCP "Bytes of the Big Apple"** (not Socrata): sidewalks, LION centerlines — resolved by fetching the DCP open-data distribution page and locating the current download link by name.

Does **not** download data — see `s1_ingest.ipynb`.

**Accept when:** every required layer (trees-primary, sidewalks, LION, borough boundary) resolves with status `success`. `street_trees_alt` is informational only (§3: "use only if the 2015 census proves unusable") and does not gate acceptance.

In [ ]:
import sys
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any

import requests
import yaml

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
print(f"s0: Resolving data sources...\nTimestamp: {datetime.now().isoformat()}\n")

## Socrata resolution (NYC Open Data catalog API)

In [ ]:
def resolve_socrata_dataset(dataset_title: str, expected_columns: list[str]) -> dict[str, Any]:
    """Resolve a Socrata dataset from NYC Open Data by title via the live catalog API."""
    catalog_url = "https://api.us.socrata.com/api/catalog/v1"
    try:
        resp = requests.get(
            catalog_url,
            params={"search_context": "data.cityofnewyork.us", "q": dataset_title, "limit": 5},
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return {"status": "error", "message": f"No Socrata dataset found for title: {dataset_title}"}

        resource = results[0].get("resource", {})
        dataset_id = resource.get("id")
        name = resource.get("name")
        if not dataset_id:
            return {"status": "error", "message": f"Socrata result found but no dataset ID: {name}"}

        schema_resp = requests.get(f"https://data.cityofnewyork.us/api/views/{dataset_id}", timeout=15)
        schema_resp.raise_for_status()
        columns = {
            c["name"]: c.get("dataTypeName", "unknown")
            for c in schema_resp.json().get("columns", [])
            if c.get("name")
        }

        missing = [c for c in expected_columns if c not in columns]
        status = "warning" if missing else "success"
        result = {
            "status": status,
            "dataset_id": dataset_id,
            "name": name,
            "url": f"https://data.cityofnewyork.us/resource/{dataset_id}.geojson",
            "columns": columns,
        }
        if missing:
            result["message"] = f"Dataset {dataset_id} missing expected columns: {missing}"
        return result

    except requests.RequestException as e:
        return {"status": "error", "message": f"Network error resolving {dataset_title}: {e}"}
    except (json.JSONDecodeError, KeyError) as e:
        return {"status": "error", "message": f"Parse error resolving {dataset_title}: {e}"}

## DCP "Bytes of the Big Apple" resolution

Sidewalks and LION are not Socrata datasets — CLAUDE.md §3 says they ship via DCP's "Bytes of the Big Apple" distribution page. Fetch that page and locate the current download link by name instead of guessing a static URL, so a catalog refresh doesn't silently break this stage.

In [ ]:
DCP_BYTES_URL = "https://www.nyc.gov/site/planning/data-maps/open-data/dwn-selected.page"


def resolve_dcp_dataset(search_terms: list[str]) -> dict[str, Any]:
    """Resolve a dataset from the DCP Bytes of the Big Apple page by matching link text."""
    try:
        resp = requests.get(DCP_BYTES_URL, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException as e:
        return {"status": "error", "message": f"Could not fetch DCP distribution page: {e}"}

    # Match <a href="...">link text containing all search_terms</a>, case-insensitive
    links = re.findall(r'<a[^>]+href="([^"]+)"[^>]*>([^<]+)</a>', resp.text, re.IGNORECASE)
    matches = [
        (href, text.strip())
        for href, text in links
        if all(term.lower() in text.lower() for term in search_terms)
    ]
    if not matches:
        return {
            "status": "error",
            "message": f"No link on {DCP_BYTES_URL} matched terms {search_terms}. "
                       "Resolve manually and record in data/SOURCES.md — do not guess a URL.",
        }

    href, text = matches[0]
    url = href if href.startswith("http") else f"https://www.nyc.gov{href}"
    return {"status": "success", "url": url, "name": text, "source_page": DCP_BYTES_URL}

## Run resolution for every source in CLAUDE.md §3

In [ ]:
sources_socrata = {
    "street_trees_primary": {
        "title": "2015 Street Tree Census (Tree Data)",
        "expected_columns": ["tree_id", "status", "tree_dbh", "spc_latin", "latitude", "longitude"],
        "required": True,
    },
    "street_trees_alt": {
        "title": "Forestry Tree Points",
        "expected_columns": ["status", "dbh", "genusspecies"],
        "required": False,
    },
    "borough_boundary": {
        "title": "Borough Boundaries",
        "expected_columns": ["boro_code", "the_geom"],
        "required": True,
    },
}

sources_dcp = {
    "sidewalks": {"search_terms": ["Sidewalk"], "required": True},
    "street_centerlines": {"search_terms": ["LION"], "required": True},
}

resolutions: dict[str, dict] = {}

for key, info in sources_socrata.items():
    print(f"Resolving (Socrata): {key}...")
    result = resolve_socrata_dataset(info["title"], info["expected_columns"])
    result["required"] = info["required"]
    resolutions[key] = result
    print(f"  status={result.get('status')} " + (result.get("message") or result.get("url", "")))

for key, info in sources_dcp.items():
    print(f"Resolving (DCP): {key}...")
    result = resolve_dcp_dataset(info["search_terms"])
    result["required"] = info["required"]
    resolutions[key] = result
    print(f"  status={result.get('status')} " + (result.get("message") or result.get("url", "")))

## QA summary and write `data/SOURCES.md`

In [ ]:
def print_qa_summary(resolutions: dict[str, dict]) -> tuple[int, int, int]:
    print("\n" + "=" * 70)
    print("S0 — Resolve sources: QA Summary")
    print("=" * 70)
    success = sum(1 for r in resolutions.values() if r["status"] == "success")
    warning = sum(1 for r in resolutions.values() if r["status"] == "warning")
    error = sum(1 for r in resolutions.values() if r["status"] == "error")
    print(f"Resolutions attempted: {len(resolutions)}")
    print(f"  success: {success}  warning: {warning}  error: {error}")
    for key, r in resolutions.items():
        if r["status"] != "success":
            flag = "REQUIRED" if r.get("required") else "optional"
            print(f"  [{r['status']}] {key} ({flag}): {r.get('message')}")
    print("=" * 70 + "\n")
    return success, warning, error


success, warning, error = print_qa_summary(resolutions)

sources_file = PROJECT_ROOT / "data" / "SOURCES.md"
resolution_summary = f"\n## Resolution Results\n\n**Resolved:** {datetime.now().isoformat()}\n"
for key, result in resolutions.items():
    resolution_summary += f"\n### {key}\n"
    resolution_summary += f"**Status:** {result.get('status')}\n"
    if result.get("dataset_id"):
        resolution_summary += f"**Dataset ID:** {result.get('dataset_id')}\n"
    if result.get("url"):
        resolution_summary += f"**URL:** {result.get('url')}\n"
    if result.get("message"):
        resolution_summary += f"**Message:** {result.get('message')}\n"
    if result.get("columns"):
        resolution_summary += f"**Columns:** {', '.join(result['columns'].keys())}\n"

existing = sources_file.read_text()
# Replace prior 'Resolution Results' section if this notebook has already been run once
existing = re.split(r"\n## Resolution Results\n", existing)[0]
sources_file.write_text(existing + resolution_summary)
print(f"Updated {sources_file}")

required_errors = [k for k, r in resolutions.items() if r.get("required") and r["status"] == "error"]
if required_errors:
    print(f"\nFAILED — required sources unresolved: {required_errors}")
    print("Fix errors above before proceeding to S1. Do not guess IDs.")
else:
    print("\nS0 complete — all required sources resolved. Ready for S1 (ingest).")